In [4]:
import re
import urllib.request
from collections import Counter
from pathlib import Path


import tiktoken

In [5]:
try:
    ROOT = Path(__file__).resolve().parent
except NameError:  # exécution dans une cellule Jupyter : `__file__` n'existe pas
    ROOT = Path.cwd()

CORPUS_PATH = ROOT / "data" / "input.txt"
SHAKESPEARE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

if not CORPUS_PATH.exists():
    CORPUS_PATH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(SHAKESPEARE_URL, CORPUS_PATH)
text = CORPUS_PATH.read_text(encoding="utf-8")

# Tokenisation BPE : quand un token n'est plus un caractère.

Toute la séance a utilisé un token par caractère. C'est simple, mais une
phrase devient une longue séquence, et l'attention coûte `T²`. Les vrais
modèles de langue utilisent des **sous-mots** appris par l'algorithme **Byte
Pair Encoding** (BPE) : on part des octets, et on fusionne à répétition la
paire la plus fréquente.

## 1. Pourquoi changer

Deux extrêmes :

- **un token = un caractère** : vocabulaire minuscule (65), aucun mot
   inconnu, mais les séquences sont longues. Le GPT du script 06 voit 64
   caractères, à peine une douzaine de mots ;
- **un token = un mot** : séquences courtes, mais un vocabulaire énorme, et
  tout mot absent à l'entraînement (une faute de frappe, un nom propre, un
  mot d'une autre langue) est **impossible à représenter**.

Le BPE est entre les deux : les morceaux fréquents (`the`, `ing`, ` and`)
deviennent un seul token, les mots rares se découpent en morceaux, et un
mot jamais vu reste toujours écrivable, au pire octet par octet.

In [6]:
words = text.split()
print(f"caractères distincts : {len(set(text)):>6,}   longueur : {len(text):>9,} tokens")
print(f"mots distincts       : {len(set(words)):>6,}   longueur : {len(words):>9,} tokens")

caractères distincts :     65   longueur : 1,115,394 tokens
mots distincts       : 25,670   longueur :   202,651 tokens


## 2. Unicode et UTF-8 : les briques de départ du BPE

Le BPE fusionne des morceaux, mais il lui faut des briques de départ. Lesquelles ?

- **Les caractères du corpus** (65 pour Shakespeare) : un caractère absent,
  comme `é` ou `🤖`, ne pourra jamais être écrit. Solution mais limité si on veut réutiliser le système autre part.
- **Tous les caractères existants** : Unicode en numérote environ 150 000. C'es trop, et la liste s'allonge à chaque nouvel emoji.
- **Les octets** : pour stocker du texte, l'ordinateur écrit chaque caractère
  en **1 à 4 octets**, des nombres de 0 à 255, selon la règle **UTF-8**.
  `a` → `[97]`, `é` → `[195, 169]`, `😀` → `[240, 159, 152, 128]`.

Tout texte, dans toutes les langues, est donc une suite d'octets. Partir des
**256 octets** donne un vocabulaire minuscule, fixe, et qui peut tout écrire.
Le BPE fusionnera ensuite les suites fréquentes (par exemple les deux octets
de `é`) en un seul token.

In [7]:
for s in ["a", "é", "€", "🤖"]:
    print(f"{s!r:5} ord={ord(s):>6}   UTF-8 : {list(s.encode('utf-8'))}")

sample = "Généré par un modèle 🤖"
print(f"{sample!r} : {len(sample)} caractères, {len(sample.encode('utf-8'))} octets")

'a'   ord=    97   UTF-8 : [97]
'é'   ord=   233   UTF-8 : [195, 169]
'€'   ord=  8364   UTF-8 : [226, 130, 172]
'🤖'   ord=129302   UTF-8 : [240, 159, 164, 150]
'Généré par un modèle 🤖' : 22 caractères, 29 octets


Shakespeare est en ASCII pur : ici un octet = un caractère. Pour du
français, un texte en octets est plus long qu'en caractères. Le BPE
compensera : `é` apparaît assez souvent pour que ses deux octets fusionnent.

## 3. L'algorithme sur un exemple

Deux fonctions suffisent :

- `get_stats` compte les paires de tokens consécutifs ;
- `merge` remplace chaque occurrence d'une paire par un nouveau token.

On répète : trouver la paire la plus fréquente, lui donner un nouvel
identifiant, la fusionner partout. C'est un algorithme de **compression**.

In [8]:
def get_stats(ids: list[int], counts: Counter | None = None) -> Counter:
    """Compte les paires de tokens consécutifs.

    Args:
        ids: Séquence de tokens.
        counts: Compteur à compléter (pour cumuler plusieurs morceaux).

    Returns:
        Le compteur `{(a, b): nombre d'occurrences}`.
    """
    counts = Counter() if counts is None else counts
    counts.update(zip(ids, ids[1:]))
    return counts


def merge(ids: list[int], pair: tuple[int, int], idx: int) -> list[int]:
    """Remplace chaque occurrence de `pair` dans `ids` par le token `idx`.

    Args:
        ids: Séquence de tokens.
        pair: Paire à fusionner.
        idx: Identifiant du nouveau token.

    Returns:
        La nouvelle séquence, plus courte d'une unité par occurrence.
    """
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


# L'exemple de l'article Wikipédia « Byte pair encoding ».
toy = list("aaabdaaabac")
symbols = {c: c for c in set(toy)}
for new_symbol in "ZYX":
    pair = get_stats(toy).most_common(1)[0][0]
    toy = merge(toy, pair, new_symbol)
    symbols[new_symbol] = symbols[pair[0]] + symbols[pair[1]]
    print(f"{pair} → {new_symbol} = {symbols[new_symbol]!r:6}   {''.join(toy)}")

('a', 'a') → Z = 'aa'     ZabdZabac
('Z', 'a') → Y = 'aaa'    YbdYbac
('Y', 'b') → X = 'aaab'   XdXac


# 4. Entraîner un tokenizer sur Shakespeare

Même chose sur les octets du texte:

- `merges` : `(a, b) → nouveau token`, **dans l'ordre** où les fusions ont
  été apprises ;
- `vocab` : `token → octets`, pour décoder.

Pour que l'exécution reste rapide, on entraîne sur les 200 000 premiers
caractères et on apprend 256 fusions : vocabulaire final de 512 tokens.

In [9]:
def train_bpe(text_: str, vocab_size: int, verbose: bool = False):
    """Apprend les fusions BPE sur un texte.

    Args:
        text_: Texte d'entraînement.
        vocab_size: Taille finale du vocabulaire (≥ 256).
        verbose: Affiche les premières fusions.

    Returns:
        Le couple `(merges, vocab)`, et la liste des ratios de compression
        après chaque fusion.
    """
    ids = list(text_.encode("utf-8"))
    n_bytes = len(ids)
    merges: dict[tuple[int, int], int] = {}
    vocab = {i: bytes([i]) for i in range(256)}
    ratios = []
    for i in range(vocab_size - 256):
        stats = get_stats(ids)
        pair = max(stats, key=stats.get)
        idx = 256 + i
        ids = merge(ids, pair, idx)
        merges[pair] = idx
        vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
        ratios.append(n_bytes / len(ids))
        if verbose and i < 20:
            print(f"fusion {i + 1:>2} : {vocab[pair[0]]!r:>8} + {vocab[pair[1]]!r:<8} → {vocab[idx]!r:12} ({stats[pair]} fois)")
    return merges, vocab, ratios


train_text = text[:200_000]
merges, vocab, ratios = train_bpe(train_text, vocab_size=512, verbose=True)
print(f"ratio de compression sur le texte d'entraînement : {ratios[-1]:.2f} octets par token")

fusion  1 :     b'e' + b' '     → b'e '        (5249 fois)
fusion  2 :     b't' + b'h'     → b'th'        (4065 fois)
fusion  3 :     b't' + b' '     → b't '        (2936 fois)
fusion  4 :     b's' + b' '     → b's '        (2797 fois)
fusion  5 :     b'o' + b'u'     → b'ou'        (2529 fois)
fusion  6 :     b'd' + b' '     → b'd '        (2502 fois)
fusion  7 :     b',' + b' '     → b', '        (2385 fois)
fusion  8 :     b'e' + b'r'     → b'er'        (2128 fois)
fusion  9 :     b'a' + b'n'     → b'an'        (1850 fois)
fusion 10 :     b'i' + b'n'     → b'in'        (1745 fois)
fusion 11 :     b'y' + b' '     → b'y '        (1594 fois)
fusion 12 :     b':' + b'\n'    → b':\n'       (1585 fois)
fusion 13 :     b'e' + b'n'     → b'en'        (1516 fois)
fusion 14 :     b' ' + b'th'    → b' th'       (1454 fois)
fusion 15 :     b'o' + b'r'     → b'or'        (1450 fois)
fusion 16 :    b'\n' + b'\n'    → b'\n\n'      (1398 fois)
fusion 17 :     b'a' + b'r'     → b'ar'        (1284 foi

Les premières fusions sont les paires les plus fréquentes de l'anglais :
`e` suivi d'une espace, `th`, des fins de mots (`t `, `s `, `d `). Les
tokens s'allongent au fil des fusions, puisqu'un nouveau token peut
fusionner avec un token déjà fusionné : ` th` (fusion 14) est fait de
l'espace et de `th` (fusion 2). Et des morceaux à cheval sur la ponctuation
(`, `, `:\n`) apparaissent.

## 5. `encode` et `decode`

**decode** : concaténer les octets de chaque token, puis décoder en UTF-8.
`errors="replace"` : une suite de tokens quelconque (produite par un modèle)
peut couper un caractère multi-octets en deux ; on affiche alors `�` au lieu
de planter.

**encode** : partir des octets et rejouer les fusions **dans l'ordre où
elles ont été apprises**. À chaque étape, parmi les paires présentes, on
fusionne celle qui a été apprise en premier (le plus petit identifiant).

In [10]:
def decode(ids: list[int]) -> str:
    """Tokens → texte."""
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")


def encode(text_: str) -> list[int]:
    """Texte → tokens, en rejouant les fusions dans leur ordre d'apprentissage."""
    ids = list(text_.encode("utf-8"))
    while len(ids) >= 2:
        stats = get_stats(ids)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break  # plus aucune fusion applicable
        ids = merge(ids, pair, merges[pair])
    return ids


def show(text_: str) -> str:
    """Affiche le découpage en tokens, séparés par des barres."""
    return "|".join(decode([t]) for t in encode(text_))


sentence = "To be, or not to be, that is the question."
print(show(sentence))
print(f"{len(sentence)} caractères → {len(encode(sentence))} tokens")

val_text = text[-50_000:]
assert decode(encode(val_text)) == val_text
print(f"validation (jamais vue) : {len(val_text):,} caractères → {len(encode(val_text)):,} tokens")

french = "Le modèle a généré une réplique étonnante."
assert decode(encode(french)) == french
print(show(french))

T|o |b|e, |or |not |to |b|e, |that |is |the |q|u|es|ti|on|.
42 caractères → 18 tokens
validation (jamais vue) : 50,000 caractères → 26,512 tokens
L|e |mo|d|�|�|l|e |a |g|�|�|n|�|�|r|�|�| |un|e |r|�|�|p|li|q|u|e |�|�|t|on|n|an|te|.


L'aller-retour est exact, y compris sur du français que le tokenizer n'# jamais vu : les octets de `é` n'ont été fusionnés avec rien, ils restent 
deux tokens séparés (affichés `�` un par un, parce qu'un octet seul n'est
pas un caractère), mais rien n'est perdu. Un tokenizer entraîné sur du
français aurait appris cette fusion.

## 6. La taille du vocabulaire

Chaque fusion raccourcit les séquences. Avec la même fenêtre de `block_size`
tokens, le modèle voit donc **plus de texte**, et l'attention, qui coûte
`T²`, devient moins chère pour un texte donné.

Le prix : la table d'embeddings et la couche de sortie ont une ligne par
token. Et plus le vocabulaire est grand, plus chaque token est rare dans les
données, donc moins bien appris. GPT-2 utilise ~50 000 tokens, GPT-4 ~100 000.

In [11]:
for n_merges in [0, 16, 64, 128, 256]:
    ratio = 1.0 if n_merges == 0 else ratios[n_merges - 1]
    print(f"{256 + n_merges:>4} tokens : {ratio:.2f} octets/token   → 64 tokens ≈ {64 * ratio:.0f} caractères de texte")

 256 tokens : 1.00 octets/token   → 64 tokens ≈ 64 caractères de texte
 272 tokens : 1.23 octets/token   → 64 tokens ≈ 79 caractères de texte
 320 tokens : 1.52 octets/token   → 64 tokens ≈ 97 caractères de texte
 384 tokens : 1.74 octets/token   → 64 tokens ≈ 112 caractères de texte
 512 tokens : 2.07 octets/token   → 64 tokens ≈ 132 caractères de texte


## 7. Les bizarreries

Le modèle ne voit **jamais les lettres**, seulement les tokens. D'où des
comportements étranges des LLM, compter les
`r` de « strawberry », épeler un mot, inverser une chaîne, faire de
l'arithmétique sur des nombres découpés n'importe comment.

In [13]:
for s in ["the", " the", "The", " THE", "strawberry", "1234567", "Shakespeare"]:
    print(f"{s!r:15} → {show(s)}")

'the'           → the
' the'          →  the
'The'           → T|he
' THE'          →  |T|H|E
'strawberry'    → st|ra|w|b|er|r|y
'1234567'       → 1|2|3|4|5|6|7
'Shakespeare'   → S|ha|k|es|p|ear|e


`the` et ` the` sont **deux tokens différents**, sans rien en commun pour
le modèle ; `The` n'est même pas un token (`T|he`), et ` THE` est épelé
lettre par lettre. Le modèle doit apprendre séparément que toutes ces formes
veulent dire la même chose. Et un mot rare se découpe en morceaux arbitraires : pour voir
combien de `r` il contient, le modèle doit avoir mémorisé l'orthographe de
chaque morceau.

## 8. Le pré-découpage de GPT-2

Sans précaution, le BPE fusionne à travers les frontières de mots et de
ponctuation : `'e '`, `', '` ou `':\n'` deviennent des tokens, et un même
mot a autant de versions que de ponctuations qui peuvent le suivre.

GPT-2 découpe d'abord le texte avec une **expression régulière** (lettres,
chiffres, ponctuation, espaces), puis applique le BPE **à l'intérieur** de
chaque morceau seulement. Les fusions ne traversent plus les frontières. Le
motif officiel utilise des classes Unicode (`\p{L}`) du module `regex` ; en
voici une version ASCII avec le `re` standard.

In [14]:
GPT2_SPLIT_PATTERN = r"""'s|'t|'re|'ve|'m|'ll|'d| ?[A-Za-z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+(?!\S)|\s+"""
print(re.findall(GPT2_SPLIT_PATTERN, "Hello've world123 how's are you!!!?  Fine."))

['Hello', "'ve", ' world', '123', ' how', "'s", ' are', ' you', '!!!?', ' ', ' Fine', '.']


L'espace reste **collée au début** du mot suivant (` world`, ` how`) :
c'est pour cela que ` the` avec espace est le token courant, et `the` sans
espace le token rare. C'est aussi ce que montre tiktokenizer
(https://tiktokenizer.vercel.app).

## 9. Le vrai tokenizer de GPT

`tiktoken` est la bibliothèque d'OpenAI. Même algorithme, entraîné sur
beaucoup plus de texte, avec 50 257 tokens.

In [ ]:

enc = tiktoken.get_encoding("gpt2")
for s in [sentence, french, "strawberry"]:
    tokens = enc.encode(s)
    print(f"GPT-2 : {len(tokens):>2} tokens   {'|'.join(enc.decode([t]) for t in tokens)}")
print(f"GPT-2 sur la validation : {len(val_text) / len(enc.encode(val_text)):.2f} caractères/token")

GPT-2 : 13 tokens   To| be|,| or| not| to| be|,| that| is| the| question|.
GPT-2 : 18 tokens   Le| mod|è|le| a| g|én|é|ré| une| ré|pl|ique| é|ton|n|ante|.
GPT-2 :  3 tokens   st|raw|berry
GPT-2 sur la validation : 3.10 caractères/token
